In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, roc_auc_score
from catboost import CatBoostRegressor, Pool, CatBoostClassifier

import time
import json
from tqdm import tqdm

import torch
from torch.utils.data import Dataset
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.nn.functional as F


# Rosbank

In [2]:
ID_COL = 'cl_id'
DOWN_TARGET = 'target_flag'
test_size = 0.20
random_state = 42
gpu_devices = "0"
verbose = 0

In [3]:
down_target = pd.read_csv('/home/jovyan/sakhno/turbo-stats/data/rosbank/target.csv')
stats = pd.read_parquet('/home/jovyan/sakhno/turbo-stats/statistics/turbo_stats_rosbank.parquet')
coles_emb = pd.read_pickle('/home/jovyan/sakhno/turbo-stats/embeddings/rosbank/coles.pickle')
ntp_rnn_emb = pd.read_pickle('/home/jovyan/sakhno/turbo-stats/embeddings/rosbank/gpt_rnn.pickle')
ntp_transf_emb = pd.read_pickle('/home/jovyan/sakhno/turbo-stats/embeddings/rosbank/gpt_transf.pickle')
llm4es_emb = pd.read_pickle('/home/jovyan/sakhno/turbo-stats/embeddings/rosbank/LLM4ES.pickle')

# Analysis

In [4]:
model_name = 'coles'

In [5]:
embs = coles_emb.copy()

In [6]:
embs[ID_COL] = embs[ID_COL].astype(int)
stats[ID_COL] = stats[ID_COL].astype(int)
down_target[ID_COL] = down_target[ID_COL].astype(int)

df = embs.merge(stats, on=ID_COL)
df = df.merge(down_target, on=ID_COL)

In [7]:
emb_cols = sorted([c for c in df.columns if c.startswith("emb_")])
down_target_col = DOWN_TARGET
target_cols = [
    c for c in df.columns
    if c not in set(emb_cols + [ID_COL] + [down_target_col])
    and np.issubdtype(df[c].dtype, np.number)
]


In [8]:
X = df[emb_cols].values
idx = np.arange(len(df))
idx_tr, idx_te = train_test_split(idx, test_size=test_size, random_state=random_state)
X_tr, X_te = X[idx_tr], X[idx_te]

# Embedding enrichment

In [9]:
feats4enrich = target_cols

In [10]:
len(feats4enrich)

37

In [11]:
coles_train_embeds = np.copy(X_tr)
coles_test_embeds = np.copy(X_te)


scaler = StandardScaler()
X_stats2enrich_tr = scaler.fit_transform(df[feats4enrich].values[idx_tr])
X_stats2enrich_te = scaler.transform(df[feats4enrich].values[idx_te])

train_stats2enrich = np.copy(X_stats2enrich_tr)
test_stats2enrich = np.copy(X_stats2enrich_te)

In [12]:
coles_train_embeds.shape, coles_test_embeds.shape

((4000, 1024), (1000, 1024))

In [13]:
train_stats2enrich.shape, test_stats2enrich.shape

((4000, 37), (1000, 37))

In [14]:
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, stats):

        self.embeddings = embeddings
        self.stats = stats

    def __len__(self):
        return self.embeddings.size(0)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.stats[idx]

In [15]:
class Enricher(nn.Module):
    def __init__(self, embed_dim, feat_dim, hidden_dim=256):
        """
        embed_dim: dimension of original embedding (d)
        hidden_dim: hidden size of residual MLP
        feat_dim: number of predicted features
        """
        super().__init__()

        self.encoder_net = nn.Sequential(
            nn.Linear(embed_dim+feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        self.decoder_net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim+feat_dim)
        )
        
        
    def forward(self, emb, stats):
        concat_emb = torch.cat([emb, stats], dim=1)
        latent_emb = self.encoder_net(concat_emb)
        restored_emb = self.decoder_net(latent_emb)
        return latent_emb, restored_emb

In [16]:
class EnrichmentLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.pred_loss = nn.MSELoss()

    def forward(self, pred_emb, true_emb):

        total_loss = nn.MSELoss()(pred_emb, true_emb) 
        return total_loss


In [17]:
def train_enricher(
    model,
    train_dataset,
    loss_fn,
    optimizer,
    batch_size=32,
    epochs=10,
    device="cuda:0"
):
    model = model.to(device)
    loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0

        for embeds, true_stats in loader:
            embeds = embeds.to(device)
            true_stats = true_stats.to(device)

            optimizer.zero_grad()

            e_enriched, e_preds = model(embeds, true_stats)
            loss = loss_fn(e_preds, torch.cat([embeds, true_stats], dim=1))

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(
            f"Epoch {epoch+1:03d} | "
            f"Total: {total_loss / len(loader):.4f} | "
        )


In [18]:
N, embed_dim, feat_dim = coles_train_embeds.shape[0], coles_train_embeds.shape[1], train_stats2enrich.shape[1]

train_embeddings = torch.tensor(coles_train_embeds, dtype=torch.float32) 
test_embeddings =  torch.tensor(coles_test_embeds, dtype=torch.float32)
train_stats = torch.tensor(train_stats2enrich, dtype=torch.float32)
test_stats = torch.tensor(test_stats2enrich, dtype=torch.float32)

train_dataset = EmbeddingDataset(train_embeddings, train_stats)

model = Enricher(
    embed_dim,
    feat_dim
)

loss_fn = EnrichmentLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

train_enricher(
    model=model,
    train_dataset=train_dataset,
    loss_fn=loss_fn,
    optimizer=optimizer,
    batch_size=16,
    epochs=130,
    device="cuda:0"
)


Epoch 001 | Total: 0.1351 | 
Epoch 002 | Total: 0.0828 | 
Epoch 003 | Total: 0.0697 | 
Epoch 004 | Total: 0.0605 | 
Epoch 005 | Total: 0.0543 | 
Epoch 006 | Total: 0.0495 | 
Epoch 007 | Total: 0.0456 | 
Epoch 008 | Total: 0.0423 | 
Epoch 009 | Total: 0.0395 | 
Epoch 010 | Total: 0.0370 | 
Epoch 011 | Total: 0.0348 | 
Epoch 012 | Total: 0.0328 | 
Epoch 013 | Total: 0.0310 | 
Epoch 014 | Total: 0.0294 | 
Epoch 015 | Total: 0.0281 | 
Epoch 016 | Total: 0.0269 | 
Epoch 017 | Total: 0.0259 | 
Epoch 018 | Total: 0.0249 | 
Epoch 019 | Total: 0.0241 | 
Epoch 020 | Total: 0.0233 | 
Epoch 021 | Total: 0.0226 | 
Epoch 022 | Total: 0.0220 | 
Epoch 023 | Total: 0.0214 | 
Epoch 024 | Total: 0.0208 | 
Epoch 025 | Total: 0.0203 | 
Epoch 026 | Total: 0.0198 | 
Epoch 027 | Total: 0.0193 | 
Epoch 028 | Total: 0.0189 | 
Epoch 029 | Total: 0.0185 | 
Epoch 030 | Total: 0.0180 | 
Epoch 031 | Total: 0.0176 | 
Epoch 032 | Total: 0.0173 | 
Epoch 033 | Total: 0.0169 | 
Epoch 034 | Total: 0.0166 | 
Epoch 035 | To

In [19]:
model.eval()
with torch.no_grad():
    train_embeds_enriched, _ = model(train_embeddings.to('cuda:0'), train_stats.to('cuda:0'))
    test_embeds_enriched, _, = model(test_embeddings.to('cuda:0'), test_stats.to('cuda:0'))

In [20]:
train_embeds_enriched = train_embeds_enriched.detach().cpu().numpy()
test_embeds_enriched = test_embeds_enriched.detach().cpu().numpy()

In [21]:
train_embeds_enriched.shape, test_embeds_enriched.shape

((4000, 256), (1000, 256))

In [22]:
classifier_model = CatBoostClassifier(logging_level='Silent', 
                                        depth=7,
                                        learning_rate=0.03,
                                        l2_leaf_reg=3,
                                        iterations=1000,
                                        random_state=random_state, 
                                        allow_writing_files=False)

In [23]:
y_down = df[DOWN_TARGET].values.astype(int)
y_tr_down, y_te_down = y_down[idx_tr], y_down[idx_te]

In [24]:
classifier_model.fit(Pool(train_embeds_enriched, y_tr_down), eval_set=Pool(test_embeds_enriched, y_te_down), use_best_model=True)
test_pred_down = classifier_model.predict(test_embeds_enriched)
test_proba_down = classifier_model.predict_proba(test_embeds_enriched)

acc = accuracy_score(y_te_down, test_pred_down)
roc_auc = roc_auc_score(y_te_down, test_proba_down[:, 1])

In [25]:
acc, roc_auc

(0.741, 0.8196115349504433)

In [26]:
df_train_enriched = pd.DataFrame({**{'cl_id': df.iloc[idx_tr]['cl_id'].values}, **{df.columns[j+1]: train_embeds_enriched[:, j]
                                                               for j in range(256)}})
df_test_enriched = pd.DataFrame({**{'cl_id': df.iloc[idx_te]['cl_id'].values}, **{df.columns[j+1]: test_embeds_enriched[:, j]
                                                               for j in range(256)}})

In [27]:
df_embeds_enriched = pd.concat([df_train_enriched, df_test_enriched], axis=0)

In [28]:
df_embeds_enriched = df_embeds_enriched.reset_index(drop=True)

In [29]:
df_embeds_enriched.shape

(5000, 257)

In [30]:
# df_embeds_enriched.to_pickle('/home/jovyan/kovtun/turbo-stats/enriched_embeds/rosbank/coles_enriched.pickle')